In [20]:
# =====================================================================
# STUDENT PERFORMANCE - REGRESSION MODELS (no plots, text output only)
# Run SECTION 0 -> 3 once, then run any model section (4-11) on its own.
# Install if needed:  pip install pandas numpy scikit-learn openpyxl
# =====================================================================

# ---------------------------------------------------------------------
# SECTION 0: IMPORTS + DATA LOADING (done ONE time only)
# ---------------------------------------------------------------------
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
df = pd.read_excel("C:/Users/hp/Downloads/student_performance.xlsx")
df.head()
# ---------------------------------------------------------------------
# SECTION 1: DATA PREPROCESSING
# ---------------------------------------------------------------------
df = df.drop(columns="Student_ID")   # ID is only a label, carries no signal -> drop
print("Missing values:\n", df.isna().sum())
print("Duplicates:", df.duplicated().sum())
print(df.describe().T)
# Result: data is clean (no missing values, no duplicates) -> no imputation 


Missing values:
 Hours_Studied     0
Attendance_Pct    0
Previous_Score    0
Sleep_Hours       0
Practice_Tests    0
Final_Score       0
dtype: int64
Duplicates: 0
                count       mean        std   min     25%    50%     75%  \
Hours_Studied   300.0   5.378000   2.608011   1.1   3.000   5.30   7.725   
Attendance_Pct  300.0  75.263667  14.215691  50.3  63.500  75.65  87.425   
Previous_Score  300.0  64.832667  17.900199  35.1  49.450  65.10  80.275   
Sleep_Hours     300.0   6.559667   1.463836   4.0   5.300   6.50   7.900   
Practice_Tests  300.0   5.010000   3.230794   0.0   2.000   5.00   8.000   
Final_Score     300.0  63.900333   9.663867  31.0  57.575  64.00  70.700   

                  max  
Hours_Studied     9.9  
Attendance_Pct  100.0  
Previous_Score   94.9  
Sleep_Hours       9.0  
Practice_Tests   10.0  
Final_Score      87.5  


In [15]:
# ---------------------------------------------------------------------
# SECTION 1: DATA PREPROCESSING
# ---------------------------------------------------------------------
df = df.drop(columns="Student_ID")   # ID is only a label, carries no signal -> drop
print("Missing values:\n", df.isna().sum())
print("Duplicates:", df.duplicated().sum())
print(df.describe().T)
# Result: data is clean (no missing values, no duplicates) -> no imputation needed.

Missing values:
 Hours_Studied     0
Attendance_Pct    0
Previous_Score    0
Sleep_Hours       0
Practice_Tests    0
Final_Score       0
dtype: int64
Duplicates: 0
                count       mean        std   min     25%    50%     75%  \
Hours_Studied   300.0   5.378000   2.608011   1.1   3.000   5.30   7.725   
Attendance_Pct  300.0  75.263667  14.215691  50.3  63.500  75.65  87.425   
Previous_Score  300.0  64.832667  17.900199  35.1  49.450  65.10  80.275   
Sleep_Hours     300.0   6.559667   1.463836   4.0   5.300   6.50   7.900   
Practice_Tests  300.0   5.010000   3.230794   0.0   2.000   5.00   8.000   
Final_Score     300.0  63.900333   9.663867  31.0  57.575  64.00  70.700   

                  max  
Hours_Studied     9.9  
Attendance_Pct  100.0  
Previous_Score   94.9  
Sleep_Hours       9.0  
Practice_Tests   10.0  
Final_Score      87.5  


In [22]:
# ---------------------------------------------------------------------
# SECTION 1: DATA PREPROCESSING
print("Missing values:\n", df.isna().sum())
print("Duplicates:", df.duplicated().sum())
print(df.describe().T)
# Result: data is clean (no missing values, no duplicates) -> no imputation needed.


Missing values:
 Hours_Studied     0
Attendance_Pct    0
Previous_Score    0
Sleep_Hours       0
Practice_Tests    0
Final_Score       0
dtype: int64
Duplicates: 0
                count       mean        std   min     25%    50%     75%  \
Hours_Studied   300.0   5.378000   2.608011   1.1   3.000   5.30   7.725   
Attendance_Pct  300.0  75.263667  14.215691  50.3  63.500  75.65  87.425   
Previous_Score  300.0  64.832667  17.900199  35.1  49.450  65.10  80.275   
Sleep_Hours     300.0   6.559667   1.463836   4.0   5.300   6.50   7.900   
Practice_Tests  300.0   5.010000   3.230794   0.0   2.000   5.00   8.000   
Final_Score     300.0  63.900333   9.663867  31.0  57.575  64.00  70.700   

                  max  
Hours_Studied     9.9  
Attendance_Pct  100.0  
Previous_Score   94.9  
Sleep_Hours       9.0  
Practice_Tests   10.0  
Final_Score      87.5  


In [23]:
# ---------------------------------------------------------------------
# SECTION 3: FEATURE ENGINEERING + TRAIN/TEST SPLIT + SCALING
# ---------------------------------------------------------------------
df["Sleep_Gap"] = (df["Sleep_Hours"] - 7).abs()                          # distance from ideal 7 h
df["Study_x_Attend"] = df["Hours_Studied"] * df["Attendance_Pct"] / 100  # effective study time
# Justification: Sleep is inverted-U, so |sleep-7| makes it linear; the interaction
# term says study hours count more when the student actually attends class.

TARGET = "Final_Score"
BASE = ["Hours_Studied", "Attendance_Pct", "Previous_Score", "Sleep_Hours", "Practice_Tests"]
ALL = BASE + ["Sleep_Gap", "Study_x_Attend"]

train, test = train_test_split(df, test_size=0.2, random_state=42)   # same split for ALL models
y_train, y_test = train[TARGET], test[TARGET]
scaler = StandardScaler().fit(train[ALL])            # fit on TRAIN only -> no data leakage
Xtr, Xte = scaler.transform(train[ALL]), scaler.transform(test[ALL])
Xtr_df, Xte_df = pd.DataFrame(Xtr, columns=ALL), pd.DataFrame(Xte, columns=ALL)

results = {}
def evaluate(name, model, Xa, Xb):
    """Fit, predict, print and store R2 / RMSE / MAE on the test set."""
    model.fit(Xa, y_train); p = model.predict(Xb)
    results[name] = {"R2": r2_score(y_test, p),
                     "RMSE": np.sqrt(mean_squared_error(y_test, p)),
                     "MAE": mean_absolute_error(y_test, p)}
    print(name, {k: round(float(v), 3) for k, v in results[name].items()})
    return model


In [24]:
# ---------------------------------------------------------------------
# SECTION 4: SIMPLE LINEAR REGRESSION (1 feature)
# Structure:   Previous_Score --> [ y = b0 + b1*x ] --> Final_Score
# ---------------------------------------------------------------------
# Justification: Previous_Score has the highest correlation -> best single predictor.
simple = evaluate("Simple Linear", LinearRegression(), train[["Previous_Score"]], test[["Previous_Score"]])
print("Intercept:", simple.intercept_, "Slope:", simple.coef_[0])


Simple Linear {'R2': 0.369, 'RMSE': 8.295, 'MAE': 6.674}
Intercept: 44.264949266187756 Slope: 0.30179652039065075


In [25]:
# ---------------------------------------------------------------------
# SECTION 5: MULTIPLE LINEAR REGRESSION (OLS, all features)
# Structure:   x1..xn --> [ y = b0 + b1*x1 + ... + bn*xn ] --> Final_Score
# ---------------------------------------------------------------------
# Justification: several factors drive the score together; OLS is an interpretable baseline.
ols = evaluate("Multiple Linear (OLS)", LinearRegression(), Xtr_df, Xte_df)
print(pd.Series(ols.coef_, index=ALL).round(3))

Multiple Linear (OLS) {'R2': 0.921, 'RMSE': 2.929, 'MAE': 2.264}
Hours_Studied     5.726
Attendance_Pct    4.113
Previous_Score    6.397
Sleep_Hours       0.400
Practice_Tests    3.228
Sleep_Gap        -1.480
Study_x_Attend   -1.089
dtype: float64


In [26]:
# ---------------------------------------------------------------------
# SECTION 6: POLYNOMIAL REGRESSION
# Structure:   x --> [ x, x^2, x^3, interactions ] --> Linear Regression --> y
# ---------------------------------------------------------------------
# Justification: Hours_Studied is curved; polynomial terms bend the line. Degree chosen by CV.
poly_gs = GridSearchCV(make_pipeline(PolynomialFeatures(include_bias=False), LinearRegression()),
                       {"polynomialfeatures__degree": [1, 2, 3]}, cv=5, scoring="r2")
poly_gs.fit(Xtr_df[BASE], y_train); print("Best degree:", poly_gs.best_params_)
poly = evaluate("Polynomial", poly_gs.best_estimator_, Xtr_df[BASE], Xte_df[BASE])

Best degree: {'polynomialfeatures__degree': 2}
Polynomial {'R2': 0.937, 'RMSE': 2.624, 'MAE': 1.995}


In [27]:
# ---------------------------------------------------------------------
# SECTION 7: SUPPORT VECTOR REGRESSION (SVR)
# Structure:   x --> [ RBF kernel ] --> fit inside an epsilon-tube
#              (errors smaller than epsilon are ignored) --> y
# ---------------------------------------------------------------------
# Justification: RBF captures non-linear curves; C = penalty, epsilon = tube width,
# gamma = kernel reach. Scaled inputs are REQUIRED (done in Section 3).
svr_gs = GridSearchCV(SVR(kernel="rbf"),
                      {"C": [1, 10, 100], "epsilon": [0.1, 1], "gamma": ["scale", 0.01, 0.1]},
                      cv=5, scoring="r2")
svr_gs.fit(Xtr, y_train); print("Best SVR:", svr_gs.best_params_)
svr = evaluate("SVR", svr_gs.best_estimator_, Xtr, Xte)

# ---------------------------------------------------------------

Best SVR: {'C': 100, 'epsilon': 0.1, 'gamma': 0.01}
SVR {'R2': 0.929, 'RMSE': 2.773, 'MAE': 2.194}


In [28]:
# SECTION 8: DECISION TREE REGRESSION
# Structure:        [Previous_Score <= t1?]
#                      yes/          \no
#           [Hours_Studied <= t2?]  [Practice_Tests <= t3?]
#              /      \                /        \
#          leaf=mean leaf=mean     leaf=mean  leaf=mean   (leaf = average score)
# ---------------------------------------------------------------------
# Justification: rule-based splits handle curves/interactions; max_depth limits overfitting.
dt_gs = GridSearchCV(DecisionTreeRegressor(random_state=42),
                     {"max_depth": [2, 3, 4, 5, 6], "min_samples_leaf": [5, 10, 20]},
                     cv=5, scoring="r2")
dt_gs.fit(train[ALL], y_train); print("Best tree:", dt_gs.best_params_)
dt = evaluate("Decision Tree", dt_gs.best_estimator_, train[ALL], test[ALL])
print(export_text(DecisionTreeRegressor(max_depth=3, random_state=42).fit(train[ALL], y_train),
                  feature_names=ALL))                # the tree printed as text rules

Best tree: {'max_depth': 5, 'min_samples_leaf': 5}
Decision Tree {'R2': 0.743, 'RMSE': 5.291, 'MAE': 4.228}
|--- Study_x_Attend <= 3.86
|   |--- Previous_Score <= 62.70
|   |   |--- Study_x_Attend <= 2.25
|   |   |   |--- value: [48.79]
|   |   |--- Study_x_Attend >  2.25
|   |   |   |--- value: [58.03]
|   |--- Previous_Score >  62.70
|   |   |--- Study_x_Attend <= 1.89
|   |   |   |--- value: [59.89]
|   |   |--- Study_x_Attend >  1.89
|   |   |   |--- value: [66.01]
|--- Study_x_Attend >  3.86
|   |--- Previous_Score <= 71.15
|   |   |--- Previous_Score <= 52.85
|   |   |   |--- value: [61.58]
|   |   |--- Previous_Score >  52.85
|   |   |   |--- value: [67.67]
|   |--- Previous_Score >  71.15
|   |   |--- Practice_Tests <= 5.50
|   |   |   |--- value: [71.75]
|   |   |--- Practice_Tests >  5.50
|   |   |   |--- value: [78.41]



In [29]:
# ---------------------------------------------------------------------
# SECTION 9: RANDOM FOREST REGRESSION
# Structure:   Data --> bootstrap 1 --> Tree 1 --+
#                  --> bootstrap 2 --> Tree 2 --+--> AVERAGE of all trees --> y
#                  --> bootstrap n --> Tree n --+   (random feature subset per split)
# ---------------------------------------------------------------------
# Justification: averaging many de-correlated trees cuts variance -> more stable than one tree.
rf_gs = GridSearchCV(RandomForestRegressor(n_estimators=200, random_state=42),
                     {"max_depth": [3, 5, None], "min_samples_leaf": [1, 5]}, cv=5, scoring="r2")
rf_gs.fit(train[ALL], y_train); print("Best RF:", rf_gs.best_params_)
rf = evaluate("Random Forest", rf_gs.best_estimator_, train[ALL], test[ALL])
print(pd.Series(rf.feature_importances_, index=ALL).sort_values(ascending=False).round(3))

# ---------------------------------------------------------------------
# SECTION 10: REGULARIZATION - RIDGE (L2) and LASSO (L1) vs OLS
# Loss = Sum(y - y_hat)^2 + penalty
#   OLS  : penalty = 0
#   Ridge: penalty = alpha * Sum(b^2)   (shrinks all coefficients, none become 0)
#   Lasso: penalty = alpha * Sum|b|     (shrinks AND sets some to exactly 0)
# ---------------------------------------------------------------------
# Justification: engineered features are correlated; penalties stabilise coefficients.
# Alpha is chosen automatically by cross-validation.
ridge = evaluate("Ridge", RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5), Xtr_df, Xte_df)
lasso = evaluate("Lasso", LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, max_iter=10000), Xtr_df, Xte_df)
print("Ridge alpha:", ridge.alpha_, "| Lasso alpha:", lasso.alpha_)
print(pd.DataFrame({"OLS": ols.coef_, "Ridge": ridge.coef_, "Lasso": lasso.coef_}, index=ALL).round(3))

# ---------------------------------------------------------------------
# SECTION 11: NEURAL NETWORK (MLP Regressor)
# Structure:   Input(7) --> Hidden(32, ReLU) --> Hidden(16, ReLU) --> Output(1)
# ---------------------------------------------------------------------
# Justification: learns non-linear patterns automatically. With only 240 training rows we use
# a small network, an L2 penalty (alpha), and learning_rate_init=0.01 so Adam converges.
nn = evaluate("Neural Network",
              MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.01,
                           learning_rate_init=0.01, max_iter=3000, random_state=42),
              Xtr, Xte)

# ---------------------------------------------------------------------
# SECTION 12: MODEL PERFORMANCE COMPARISON (table)
# ---------------------------------------------------------------------
# R2 closer to 1 = better; RMSE / MAE smaller = better (units = exam marks).
summary = pd.DataFrame(results).T.sort_values("R2", ascending=False).round(3)
print(summary)
print("Best model: ", summary.index[0])

Best RF: {'max_depth': None, 'min_samples_leaf': 1}
Random Forest {'R2': 0.86, 'RMSE': 3.904, 'MAE': 3.006}
Previous_Score    0.373
Study_x_Attend    0.342
Practice_Tests    0.085
Attendance_Pct    0.074
Hours_Studied     0.061
Sleep_Gap         0.034
Sleep_Hours       0.031
dtype: float64
Ridge {'R2': 0.923, 'RMSE': 2.897, 'MAE': 2.339}
Lasso {'R2': 0.923, 'RMSE': 2.9, 'MAE': 2.294}
Ridge alpha: 4.714866363457395 | Lasso alpha: 0.04291934260128776
                  OLS  Ridge  Lasso
Hours_Studied   5.726  3.521  4.703
Attendance_Pct  4.113  3.183  3.673
Previous_Score  6.397  6.241  6.333
Sleep_Hours     0.400  0.456  0.392
Practice_Tests  3.228  3.195  3.199
Sleep_Gap      -1.480 -1.398 -1.430
Study_x_Attend -1.089  1.287  0.000
Neural Network {'R2': 0.892, 'RMSE': 3.432, 'MAE': 2.819}
                          R2   RMSE    MAE
Polynomial             0.937  2.624  1.995
SVR                    0.929  2.773  2.194
Ridge                  0.923  2.897  2.339
Lasso                  0.923 